In [51]:
import json
import re
from pathlib import Path
from tqdm import tqdm


def split_sentences(text):
    """按中文标点符号切句，保留标点"""
    sentences = re.split(r'(?<=[。！？,.?!])', text)
    return [s.strip() for s in sentences if s.strip()]


def get_supporting_fact_index(answer_start, sentences):
    """根据 answer_start 找到答案所在句子的索引"""
    char_count = 0
    for idx, sent in enumerate(sentences):
        char_count += len(sent)
        if char_count > answer_start:
            return idx
    return -1


def convert_to_examples(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    output = []
    global_id = 0

    for item in tqdm(raw_data['data']):
        for para in item.get("paragraphs", []):
            full_text = para.get("context", "").strip()
            sentences = split_sentences(full_text)
            joined_context = "".join(sentences)  # 注意这里不加空格，保持和read_examples一致
            qas = para.get("qas", [])

            for qa in qas:
                question = qa.get("question", "")
                is_impossible = qa.get("is_impossible", "false").lower() == "true"
                answers = qa.get("answers", [])

                if is_impossible or not answers:
                    example = {
                        "_id": global_id,
                        "context": [[f"p{global_id}", sentences]],
                        "question": question,
                        "answer": "yes",
                        "answer_start": -1,
                        "supporting_facts": []
                    }
                    output.append(example)
                    global_id += 1
                    continue

                for ans in answers:
                    answer_text = ans.get("text", "").strip()
                    answer_text_lower = answer_text.lower()

                    # 分类答案直接处理
                    if answer_text_lower in ["yes", "no", "unknown"]:
                        example = {
                            "_id": global_id,
                            "context": [[f"p{global_id}", sentences]],
                            "question": question,
                            "answer": answer_text.upper(),
                            "answer_start": -1,
                            "supporting_facts": []
                        }
                        output.append(example)
                        global_id += 1
                        continue

                    # 重新定位答案起始位置，保证offset准确
                    answer_start = joined_context.find(answer_text)
                    if answer_start == -1:
                        print(f"[⚠️ 找不到答案] qid={global_id} answer={answer_text}")
                        continue

                    sf_idx = get_supporting_fact_index(answer_start, sentences)
                    supporting_facts = [[f"p{global_id}", sf_idx]] if sf_idx >= 0 else []

                    example = {
                        "_id": global_id,
                        "context": [[f"p{global_id}", sentences]],
                        "question": question,
                        "answer": answer_text,
                        "answer_start": answer_start,
                        "supporting_facts": supporting_facts
                    }
                    output.append(example)
                    global_id += 1

    # 重新编号_id，确保连续
    for i, example in enumerate(output):
        example["_id"] = i

    # 打印部分分类答案验证
    debug_count = 0
    # for ex in output:
    #     if debug_count < 2 and str(ex["answer"]).strip().lower() in ["yes", "no", "unknown"]:
    #         print("🟨 检查特殊answer样本：")
    #         print(json.dumps(ex, indent=2, ensure_ascii=False))
    #         debug_count += 1

    # 打印测试样本段落示例
    test_index = 0
    if len(output) > test_index:
        for i in range(2):
            print(json.dumps(output[test_index + i], indent=2, ensure_ascii=False))

    # 写入结果文件
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"✅ 转换完成，共生成 {len(output)} 条样本：{output_file}")


if __name__ == "__main__":
    base_input = Path("./data_cail")
    base_output = Path("../data")

    # convert_to_examples(base_input / "test_ground_truth.json", base_output / "test.json")
    # convert_to_examples(base_input / "big_train_data.json", base_output / "train.json")
    convert_to_examples(base_input / "devt_ground_truth.json", base_output / "dev.json")

100%|██████████| 1000/1000 [00:00<00:00, 7576.98it/s]


{
  "_id": 0,
  "context": [
    [
      "p0",
      [
        "经审理查明:原告曹0与被告张x1原5夫妻关系2001年5月27日,",
        "原、被告生育一女张x62011年7月21日,",
        "原、被告因夫妻感情破裂,",
        "在重庆市荣昌县人民法院调解离婚,",
        "调解书载明:“原、被告之女张x6随被告张x1生活”原、被告离婚后又自行协商,",
        "约定小孩张x6暂由原告代管,",
        "被告每月向原告支付张x6的生活费800元因此,",
        "张x6一直随原告生活至今,",
        "期间被告未依照约定按时向原告支付张x6的生活费另查明:张x6现在荣昌县城西小学上学,",
        "为城镇居民户口原告曹0系重庆博耐特压铸有限公司职工,",
        "收入比较稳定,",
        "现已再婚,",
        "其与丈夫何祖君共同购买了位于荣昌县昌元街道昌州大道西段的房屋一套本案在审理过程中,",
        "本院依法对张x6作了询问笔录,",
        "张x6称从父母离婚后其一直随母亲生活,",
        "只在每年暑假到父亲家里玩耍一段时间,",
        "现在自己愿意随母亲曹0生活上述事实,",
        "有原告陈述,",
        "被告答辩状,",
        "常住人口登记卡,",
        "重庆市荣昌县人民法院作出的(2011)荣4民初字第1929号民事调解书,",
        "原、被告签订的协议,",
        "结婚证,",
        "房产证,",
        "劳动合同,",
        "重庆市基本养老保险个人账户信息表,",
        "对张x6的询问笔录等证据予以证实"
      ]
    ]
  ],
  "question": "原告曹0与被告张x1因何离婚？",
  "answer": "夫妻感情破裂",
  "answer_start": 60,
  "supporting_facts": [
    [
      "p0",
      2
  

In [47]:
import json
import re
from pathlib import Path
from tqdm import tqdm


def split_sentences(text):
    """按中文标点符号切句，保留标点"""
    sentences = re.split(r'(?<=[。！？,.?!])', text)
    return [s.strip() for s in sentences if s.strip()]


def get_supporting_fact_index(answer_start, sentences):
    """根据 answer_start 找到答案所在句子的索引"""
    char_count = 0
    for idx, sent in enumerate(sentences):
        char_count += len(sent)
        if char_count > answer_start:
            return idx
    return -1


def convert_to_examples(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    output = []
    global_id = 0

    for item in tqdm(raw_data['data']):
        for para in item.get("paragraphs", []):
            full_text = para.get("context", "").strip()
            sentences = split_sentences(full_text)
            joined_context = "".join(sentences)
            qas = para.get("qas", [])

            for qa in qas:
                question = qa.get("question", "")
                is_impossible = qa.get("is_impossible", "false").lower() == "true"
                answers = qa.get("answers", [])

                if is_impossible or not answers:
                    example = {
                        "_id": global_id,
                        "context": [[f"p{global_id}", sentences]],
                        "question": question,
                        "answer": "yes",
                        "answer_start": -1,
                        "supporting_facts": []
                    }
                    output.append(example)
                    global_id += 1
                    continue

                for ans in answers:
                    answer_text = ans.get("text", "").strip()
                    answer_text_lower = answer_text.lower()

                    # ✅ 分类答案处理
                    if answer_text_lower in ["yes", "no", "unknown"]:
                        example = {
                            "_id": global_id,
                            "context": [[f"p{global_id}", sentences]],
                            "question": question,
                            "answer": answer_text.upper(),
                            "answer_start": -1,
                            "supporting_facts": []
                        }
                        output.append(example)
                        global_id += 1
                        continue

                    # ✅ span 答案：动态查找位置
                    answer_start = joined_context.find(answer_text)
                    if answer_start == -1:
                        print(f"[⚠️ 找不到答案] qid={global_id} answer={answer_text}")
                        continue

                    sf_idx = get_supporting_fact_index(answer_start, sentences)
                    supporting_facts = [[f"p{global_id}", sf_idx]] if sf_idx >= 0 else []

                    example = {
                        "_id": global_id,
                        "context": [[f"p{global_id}", sentences]],
                        "question": question,
                        "answer": answer_text,
                        "answer_start": answer_start,
                        "supporting_facts": supporting_facts
                    }
                    output.append(example)
                    global_id += 1

    # ✅ 重新编号
    for i, example in enumerate(output):
        example["_id"] = i

    # ✅ 打印分类类答案
    debug_count = 0
    # for ex in output:
    #     if debug_count < 2 and str(ex["answer"]).strip().lower() in ["yes", "no", "unknown"]:
    #         print("🟨 检查特殊answer样本：")
    #         print(json.dumps(ex, indent=2, ensure_ascii=False))
    #         debug_count += 1

    # ✅ 打印任意 test 样本
    test_index = 0
    if len(output) > test_index:
        for i in range(4):
            print(json.dumps(output[test_index + i], indent=2, ensure_ascii=False))

    # ✅ 写出文件
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"✅ 转换完成，共生成 {len(output)} 条样本：{output_file}")


if __name__ == "__main__":
    base_input = Path("./data_cail")
    base_output = Path("../data")

    convert_to_examples(base_input / "test_ground_truth.json", base_output / "test.json")
    # convert_to_examples(base_input / "big_train_data.json", base_output / "train.json")
    # convert_to_examples(base_input / "devt_ground_truth.json", base_output / "dev.json")

100%|██████████| 1000/1000 [00:00<00:00, 5582.93it/s]


🟨 检查特殊answer样本：
{
  "_id": 6,
  "context": [
    [
      "p6",
      [
        "经审理查明:原告刘x0与被告刘x1原6夫妻关系,",
        "双方婚姻关系存续期间生育一子刘×3(曾用名:刘×4,",
        "2002年9月24日出生)2009年6月5日,",
        "刘x0与刘x1经北京市通州区人民法院调解离婚,",
        "关于子女抚养,",
        "双方约定男孩刘×3由刘x1自行抚养二人离婚后,",
        "刘×3随刘x1生活,",
        "并在刘x1的原籍读书至小学四年级,",
        "后刘x0为刘×3办理转学到本市通州区读书,",
        "后刘×3随刘x0生活至今因刘×3已年满十周岁,",
        "本院依法征求了刘×3对于变更抚养关系的意见,",
        "刘×3称其与父母的关系均很好,",
        "现在北京读书与母亲刘x0共同生活,",
        "寒暑假与父亲生活,",
        "在北京的生活比较习惯,",
        "愿意随母亲刘x0生活在本案审理过程中,",
        "本院依法向刘x1送达了起诉书及开庭传票后,",
        "刘x1未到庭参加诉讼,",
        "后向本庭邮寄了一份书面意见,",
        "内容为:“本人刘x1同意变更儿子刘×4(刘×3)抚养权问题,",
        "同意抚养权变更为刘x0抚养”本庭向刘x0出示了上述书面意见,",
        "刘x0表示无异议以上事实,",
        "有(2009)通民初字第8185号民事调解书、出生证明、独生子女证、户口登记簿、刘x1书面意见、谈话笔录、开庭笔录等证据在案佐证"
      ]
    ]
  ],
  "question": "刘x1为什么没有到法庭参加诉讼？",
  "answer": "yes",
  "answer_start": -1,
  "supporting_facts": []
}
🟨 检查特殊answer样本：
{
  "_id": 7,
  "context": [
    [
      "p7

In [46]:
import json  # 可运行版本
import re
from pathlib import Path
from tqdm import tqdm


def split_sentences(text):
    """按中文标点符号切句，保留标点"""
    sentences = re.split(r'(?<=[。！？,.?!])', text)
    return [s.strip() for s in sentences if s.strip()]


def get_supporting_fact_index(answer_start, sentences):
    """根据 answer_start 找到答案所在句子的索引"""
    char_count = 0
    for idx, sent in enumerate(sentences):
        char_count += len(sent)
        if char_count > answer_start:
            return idx
    return -1


def convert_to_examples(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    output = []
    global_id = 0  # 全局计数器，唯一编号

    for item in tqdm(raw_data['data']):
        for para in item.get("paragraphs", []):
            full_text = para.get("context", "").strip()
            sentences = split_sentences(full_text)
            qas = para.get("qas", [])

            for qa in qas:
                question = qa.get("question", "")
                is_impossible = qa.get("is_impossible", "false").lower() == "true"
                answers = qa.get("answers", [])

                if is_impossible or not answers:
                    example = {
                        "_id": global_id,
                        "context": [[f"p{global_id}", sentences]],
                        "question": question,
                        "answer": "yes",
                        "answer_start": -1,
                        "supporting_facts": []
                    }
                    output.append(example)
                    global_id += 1
                else:
                    for ans in answers:
                        answer_text = ans.get("text", "").strip().lower()
                        answer_start = ans.get("answer_start", -1)
                    
                        # ✅ 分类类问题处理：YES / NO / UNKNOWN
                        if answer_text in ["yes", "no", "unknown"]:
                            example = {
                                "_id": global_id,
                                "context": [[f"p{global_id}", sentences]],
                                "question": question,
                                "answer": answer_text.upper(),  # 保持格式统一
                                "answer_start": -1,
                                "supporting_facts": []
                            }
                            output.append(example)
                            global_id += 1
                            continue
                    
                        # ✅ 正常 span 答案处理
                        if answer_start >= len(full_text):
                            print(f"[⚠️ answer 超出范围] case_id={global_id}, start={answer_start}, len(context)={len(full_text)}")
                            continue
                    
                        sf_idx = get_supporting_fact_index(answer_start, sentences)
                        supporting_facts = [[f"p{global_id}", sf_idx]] if sf_idx >= 0 else []
                    
                        example = {
                            "_id": global_id,
                            "context": [[f"p{global_id}", sentences]],
                            "question": question,
                            "answer": ans.get("text", ""),
                            "answer_start": answer_start,
                            "supporting_facts": supporting_facts
                        }
                        output.append(example)
                        global_id += 1
                    # for ans in answers:
                    #     answer_text = ans.get("text", "")
                    #     answer_start = ans.get("answer_start", -1)

                    #     # 修复 answer_start 越界情况
                    #     if answer_start >= len(full_text):
                    #         print(f"[⚠️ answer 超出范围] case_id={global_id}, start={answer_start}, len(context)={len(full_text)}")
                    #         continue

                    #     sf_idx = get_supporting_fact_index(answer_start, sentences)
                    #     supporting_facts = [[f"p{global_id}", sf_idx]] if sf_idx >= 0 else []

                    #     example = {
                    #         "_id": global_id,
                    #         "context": [[f"p{global_id}", sentences]],
                    #         "question": question,
                    #         "answer": answer_text,
                    #         "answer_start": answer_start,
                    #         "supporting_facts": supporting_facts
                    #     }
                    #     output.append(example)
                    #     global_id += 1

    # ✅ 修复：确保所有样本 _id 连续编号
    for i, example in enumerate(output):
        example["_id"] = i

    # 打印样例检查
    debug_count = 0
    for i, example in enumerate(output):
        example["_id"] = i
        if debug_count < 2 and str(example["answer"]).strip().lower() in ["yes", "no", "unknown"]:
            print("🟨 检查特殊answer样本：")
            print(json.dumps(example, indent=2, ensure_ascii=False))
            debug_count += 1
    test_index = 1222
    if len(output) > test_index:
        for i in range(4):
            print(json.dumps(output[test_index + i], indent=2, ensure_ascii=False))

    # with open(output_file, 'w', encoding='utf-8') as f:
    #     json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"✅ 转换完成，共生成 {len(output)} 条样本：{output_file}")


if __name__ == "__main__":
    base_input = Path("./data_cail")
    base_output = Path("../data")

    # convert_to_examples(base_input / "big_train_data.json", base_output / "train.json")
    # convert_to_examples(base_input / "devt_ground_truth.json", base_output / "dev.json")
    convert_to_examples(base_input / "test_ground_truth.json", base_output / "test.json")

100%|██████████| 1000/1000 [00:00<00:00, 4976.13it/s]

🟨 检查特殊answer样本：
{
  "_id": 6,
  "context": [
    [
      "p6",
      [
        "经审理查明:原告刘x0与被告刘x1原6夫妻关系,",
        "双方婚姻关系存续期间生育一子刘×3(曾用名:刘×4,",
        "2002年9月24日出生)2009年6月5日,",
        "刘x0与刘x1经北京市通州区人民法院调解离婚,",
        "关于子女抚养,",
        "双方约定男孩刘×3由刘x1自行抚养二人离婚后,",
        "刘×3随刘x1生活,",
        "并在刘x1的原籍读书至小学四年级,",
        "后刘x0为刘×3办理转学到本市通州区读书,",
        "后刘×3随刘x0生活至今因刘×3已年满十周岁,",
        "本院依法征求了刘×3对于变更抚养关系的意见,",
        "刘×3称其与父母的关系均很好,",
        "现在北京读书与母亲刘x0共同生活,",
        "寒暑假与父亲生活,",
        "在北京的生活比较习惯,",
        "愿意随母亲刘x0生活在本案审理过程中,",
        "本院依法向刘x1送达了起诉书及开庭传票后,",
        "刘x1未到庭参加诉讼,",
        "后向本庭邮寄了一份书面意见,",
        "内容为:“本人刘x1同意变更儿子刘×4(刘×3)抚养权问题,",
        "同意抚养权变更为刘x0抚养”本庭向刘x0出示了上述书面意见,",
        "刘x0表示无异议以上事实,",
        "有(2009)通民初字第8185号民事调解书、出生证明、独生子女证、户口登记簿、刘x1书面意见、谈话笔录、开庭笔录等证据在案佐证"
      ]
    ]
  ],
  "question": "刘x1为什么没有到法庭参加诉讼？",
  "answer": "yes",
  "answer_start": -1,
  "supporting_facts": []
}
🟨 检查特殊answer样本：
{
  "_id": 7,
  "context": [
    [
      "p7